# DINOv3 Patch Similarity — Segmentation without SAM

DINOv3 encodes every 16×16-pixel patch as a 768-dim vector. Semantically similar patches
(e.g. apple surfaces across different images) cluster together in this space.

**Core idea:** build a single *prototype* vector for a target object, slide it over every
patch in a scene via cosine similarity, upsample the 14×14 result to full resolution,
and threshold → binary segmentation mask.

**Three query modes** (each has its own section below):

| Mode | Prototype source | When to use |
|---|---|---|
| **Gallery** | Mean foreground patch from reference images | Labelled examples available |
| **Bounding box** | Mean of patches inside a drawn box on the scene | Semi-interactive |
| **Point** | Single patch at a clicked pixel | Fastest, least robust |

All heavy functions live in `dinov3_seg_utils.py` — this notebook only contains
configuration variables and calls to those functions.

**Prerequisites:** Fruits360 dataset extracted, `apple_pear_1.jpg` scene image,
`transformers >= 4.56.0`, `torch`, `pillow`, `matplotlib`, `scikit-learn`.

## Setup

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch
import sys; sys.path.insert(0, ".")

from dinov3_seg_utils import (
    PATCH_SIZE, IMG_SIZE, N_SIDE, N_PATCH,
    load_model, get_patch_tokens, patch_sim_map,
    get_image_paths, find_class,
    build_gallery_prototype, build_bbox_prototype, build_point_prototype,
    compute_sim_mask,
    show_patch_sim_grid, show_detection,
    show_threshold_sweep, show_patch_grid_overlay,
)
print("Imports OK.")

In [ ]:
# ── Edit these two paths ──────────────────────────────────────────────
DATASET_PATH = pathlib.Path(r"../fruits-360-100x100")   
SCENE_PATH   = pathlib.Path(r"apple_pear_1.jpg") 
# ────────────────────────────────────────────────────────────────────────

TRAIN_ROOT = DATASET_PATH / "Training"
assert TRAIN_ROOT.exists(), f"Training/ not found under {DATASET_PATH}"
assert SCENE_PATH.exists(), f"Scene image not found: {SCENE_PATH}"

classes = sorted([d.name for d in TRAIN_ROOT.iterdir() if d.is_dir()])
print(f"Dataset: {len(classes)} classes  |  scene: {SCENE_PATH.name}")

## Authenticate with Hugging Face

DINOv3 weights are gated on Hugging Face and require a free account + access token.

1. Create an account at huggingface.co
2. Go to **Settings → Access Tokens → New token** (read permission is enough)
3. Accept the model license at `facebook/dinov3-vitb16-pretrain-lvd1689m`
4. Run the cell below and paste your token when prompted

In [ ]:
from huggingface_hub import login
token = "my_private_token_which_I_won't_share!!!!!!"
login(token=token, add_to_git_credential=False)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model, processor, NUM_REG = load_model(device=DEVICE)

In [ ]:
scene_img      = Image.open(SCENE_PATH).convert("RGB")
orig_w, orig_h = scene_img.size
scene_arr      = np.array(scene_img)

print(f"Scene: {orig_w} x {orig_h} px  |  extracting patch tokens ...")
scene_tokens = get_patch_tokens(SCENE_PATH, model, processor, DEVICE, NUM_REG)
print(f"scene_tokens: {scene_tokens.shape}")

---
## Part 1 — Dense Patch Similarity Maps

Select a query patch in one image and compute its cosine similarity to every patch in
target images. Same-class targets light up at the corresponding semantic region;
cross-class targets stay uniformly cold.

This is the mechanism the detector uses, visualised at the single-patch level.

In [ ]:
CLASS_A = "Strawberry"   # query class — change to any class in the dataset
CLASS_B = "Banana"       # cross-class targets
N_IMG   = 4              # number of target images per class

In [ ]:
CLASS_A = find_class(CLASS_A, classes)
CLASS_B = find_class(CLASS_B, classes)
print(f"Class A: {CLASS_A}  |  Class B: {CLASS_B}")

paths_a = get_image_paths(TRAIN_ROOT, CLASS_A, N_IMG + 1)
paths_b = get_image_paths(TRAIN_ROOT, CLASS_B, N_IMG)

QUERY_PATH    = paths_a[0]
SAME_PATHS    = paths_a[1:]
CROSS_PATHS   = paths_b
QUERY_PATCHES = [(3, 7), (7, 7), (10, 7)]          # top / centre / bottom
PATCH_COLORS  = ["#e74c3c", "#2ecc71", "#3498db"]

print("Extracting patch tokens ...")
query_tok    = get_patch_tokens(QUERY_PATH, model, processor, DEVICE, NUM_REG)
same_tokens  = [get_patch_tokens(p, model, processor, DEVICE, NUM_REG) for p in SAME_PATHS]
cross_tokens = [get_patch_tokens(p, model, processor, DEVICE, NUM_REG) for p in CROSS_PATHS]
print("Done.")

In [ ]:
show_patch_sim_grid(
    QUERY_PATH, query_tok, QUERY_PATCHES, PATCH_COLORS,
    SAME_PATHS, same_tokens, CROSS_PATHS, cross_tokens,
    CLASS_A, CLASS_B,
    save_path="patch_similarity_map.png",
)

---
## Part 2 — Object Detection via Cosine Similarity Thresholding

We now apply the same idea to the multi-fruit scene (`apple_pear_1.jpg`).
A single prototype vector is compared to every scene patch; the similarity
map is upsampled to original resolution and thresholded.

```
prototype  (768-dim)
    │
    ▼  cosine similarity with every scene patch
14 x 14 similarity grid
    │  bilinear upsample
    ▼
full-resolution heatmap
    │  percentile threshold
    ▼
binary segmentation mask
```

Each mode below builds the prototype differently. The detection step is identical for all.

### Mode A — Gallery

The prototype is the mean of *foreground* patch tokens extracted from a set of reference
images of the target class.

Because Fruits360 images have a plain white background, the first principal component
of the patch tokens cleanly separates fruit-surface patches from background patches —
no manual annotation needed.

In [ ]:
GALLERY_CLASS  = "Apple Red"   # prefix match — change to any Fruits360 class
N_REF          = 20            # number of reference images
SIM_PERCENTILE = 65            # keep top (100 - SIM_PERCENTILE)% of patches

In [ ]:
ref_class = find_class(GALLERY_CLASS, classes)
ref_paths = get_image_paths(TRAIN_ROOT, ref_class, N_REF)

gallery_proto, n_fg = build_gallery_prototype(
    ref_paths, model, processor, DEVICE, NUM_REG
)

# Preview reference images
n_prev = min(6, len(ref_paths))
fig, axes = plt.subplots(1, n_prev, figsize=(n_prev * 2, 2.2))
for ax, p in zip(axes, ref_paths[:n_prev]):
    ax.imshow(Image.open(p)); ax.axis("off")
plt.suptitle(f"Gallery reference: {ref_class}  ({n_fg} foreground patches)", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
sim_up, mask, thr = compute_sim_mask(
    gallery_proto, scene_tokens, SIM_PERCENTILE, orig_w, orig_h
)
show_detection(
    scene_arr, sim_up, mask, "gallery", thr, SIM_PERCENTILE,
    save_path="detection_gallery.png",
)

# Make available for the threshold sweep in Part 3
sim_upsampled = sim_up
binary_mask   = mask
LAST_MODE     = "gallery"

### Mode B — Bounding Box

The prototype is the mean of all patch tokens whose receptive fields overlap a
bounding box drawn on the scene image.

No reference images are needed — just a rough box around the object you want to find.
The box coordinates are in the original image pixel space.

In [ ]:
QUERY_BOX      = (1938, 1054, 2538, 1800)   # (x1, y1, x2, y2) in original px
SIM_PERCENTILE = 65

In [ ]:
bbox_proto, n_patches = build_bbox_prototype(
    scene_tokens, QUERY_BOX, orig_w, orig_h
)

# Preview: box overlaid on scene
x1, y1, x2, y2 = QUERY_BOX
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(scene_arr)
ax.add_patch(mpatches.Rectangle(
    (x1, y1), x2 - x1, y2 - y1,
    linewidth=2.5, edgecolor="#e74c3c", facecolor="#e74c3c", alpha=0.20))
ax.set_title(f"Query box  ({n_patches} patches)", fontsize=10)
ax.axis("off"); plt.tight_layout(); plt.show()

In [ ]:
sim_up, mask, thr = compute_sim_mask(
    bbox_proto, scene_tokens, SIM_PERCENTILE, orig_w, orig_h
)
show_detection(
    scene_arr, sim_up, mask, "bbox", thr, SIM_PERCENTILE,
    query_box=QUERY_BOX,
    save_path="detection_bbox.png",
)

sim_upsampled = sim_up
binary_mask   = mask
LAST_MODE     = "bbox"

### Mode C — Point

The prototype is a single patch token — the one whose 16×16-pixel receptive field
contains the given pixel coordinate.

This is the fastest mode but also the least robust: a single patch may not be
representative enough if the clicked pixel lands on a specular highlight,
shadow, or boundary region.

In [ ]:
QUERY_POINT    = (400, 300)   # (x, y) in original image pixels — click inside the target
SIM_PERCENTILE = 65

In [ ]:
point_proto, q_row, q_col = build_point_prototype(
    scene_tokens, QUERY_POINT, orig_w, orig_h
)

# Preview: patch highlighted on scene
pw = PATCH_SIZE / (IMG_SIZE / orig_w)
ph = PATCH_SIZE / (IMG_SIZE / orig_h)
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(scene_arr)
ax.add_patch(mpatches.Rectangle(
    (q_col * pw, q_row * ph), pw, ph,
    linewidth=2.5, edgecolor="#e74c3c", facecolor="#e74c3c", alpha=0.35))
ax.plot(QUERY_POINT[0], QUERY_POINT[1], "r+", markersize=14, markeredgewidth=2.5)
ax.set_title(f"Query point {QUERY_POINT}  \u2192  patch (row {q_row}, col {q_col})",
             fontsize=10)
ax.axis("off"); plt.tight_layout(); plt.show()

In [ ]:
sim_up, mask, thr = compute_sim_mask(
    point_proto, scene_tokens, SIM_PERCENTILE, orig_w, orig_h
)
show_detection(
    scene_arr, sim_up, mask, "point", thr, SIM_PERCENTILE,
    query_point=QUERY_POINT,
    save_path="detection_point.png",
)

sim_upsampled = sim_up
binary_mask   = mask
LAST_MODE     = "point"

---
## Part 3 — Threshold Sensitivity

`SIM_PERCENTILE` is a hard hyperparameter: too low → many false positives;
too high → missed detections. There is no universally correct value.

This cell uses whichever mode was run last (gallery / bbox / point).
Run one of the detection cells above before running this one.

In [ ]:
show_threshold_sweep(
    scene_arr, sim_upsampled,
    percentiles=[40, 55, 65, 75, 85],
    query_mode=LAST_MODE,
    save_path="threshold_sweep.png",
)

---
## Part 4 — Limitations & Why We Need SAM

| Problem | DINOv3 cosine threshold | SAM |
|---|---|---|
| **Boundary precision** | ± one patch (~16 px in 224-px space) | Sub-pixel — learned upsampling |
| **Threshold selection** | Manual, scene-dependent | Not needed — learned mask decoder |
| **Multi-instance separation** | One mask for all similar patches | Distinct mask per object instance |
| **Background bleed** | Leaks to similarly-coloured regions | Object-shape prior in decoder |
| **Confidence calibration** | Raw cosine scores | Calibrated IoU and stability scores |

In [ ]:
show_patch_grid_overlay(
    scene_arr, orig_w, orig_h,
    save_path="patch_grid_overlay.png",
)

patch_w = PATCH_SIZE / (IMG_SIZE / orig_w)
patch_h = PATCH_SIZE / (IMG_SIZE / orig_h)
print(f"Scene          : {orig_w} x {orig_h} px")
print(f"Patch grid     : {N_SIDE} x {N_SIDE} = {N_PATCH} tokens")
print(f"One patch      : {patch_w:.0f} x {patch_h:.0f} px in original image")
print(f"Min. precision : ~{max(patch_w, patch_h):.0f} px")
print()
print("SAM: 64 x 64 feature resolution + learned upsampling")
print(f"  boundary ~{orig_w / 64:.0f} x {orig_h / 64:.0f} px  (before learned upsample)")